# Spark: сравнение ресурсов

Идея: запускаем одинаковые действия на одном датасете, но меняем ресурсы Spark-приложения. Сравнивайте время выполнения, Spark UI `Jobs/Stages/Executors`, spill и количество task'ов.

## Как переключать размер кластера

Перед запуском notebook поднимите один из профилей:

```bash
docker compose --env-file profiles/spark-small.env up -d --scale spark-worker=1
docker compose --env-file profiles/spark-medium.env up -d --scale spark-worker=2
docker compose --env-file profiles/spark-large.env up -d --scale spark-worker=4
```

Spark Master UI: http://localhost:8080. Driver UI обычно: http://localhost:4040.

In [ ]:
import sys
sys.path.append("/opt/workspace/scripts")

from spark_lab import make_spark, benchmark_groupby, benchmark_join, benchmark_skew

## Профиль приложения

Меняйте `PROFILE`. Важно: если SparkSession уже создана, сначала выполните `spark.stop()`, потом создайте новую сессию.

In [ ]:
PROFILES = {
    "tiny": {
        "executor_memory": "512m",
        "executor_cores": 1,
        "cores_max": 1,
        "shuffle_partitions": 8,
    },
    "normal": {
        "executor_memory": "1g",
        "executor_cores": 1,
        "cores_max": 2,
        "shuffle_partitions": 16,
    },
    "wide": {
        "executor_memory": "2g",
        "executor_cores": 2,
        "cores_max": 4,
        "shuffle_partitions": 32,
    },
}

PROFILE = "normal"
conf = PROFILES[PROFILE]
conf

In [ ]:
spark = make_spark(app_name=f"resource_lab_{PROFILE}", **conf)
print("Spark UI:", "http://localhost:4040")
print("defaultParallelism:", spark.sparkContext.defaultParallelism)
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
BASE = "data/lab"
EVENTS = f"{BASE}/events"
EVENTS_PARTITIONED = f"{BASE}/events_partitioned"
USERS = f"{BASE}/users"
SKEWED = f"{BASE}/skewed_events"

## CPU/shuffle: group by

Здесь обычно видна разница между `cores_max=1/2/4` и количеством shuffle partitions.

In [ ]:
benchmark_groupby(spark, EVENTS)

## Join: sort-merge против broadcast

Первый запуск запрещает broadcast, второй явно broadcast'ит маленький справочник.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
benchmark_join(spark, EVENTS, USERS, broadcast=False)

In [ ]:
benchmark_join(spark, EVENTS, USERS, broadcast=True)

## Partition pruning

Сравните чтение обычного и партиционированного датасета. В Spark UI в SQL-плане должно быть видно, что читается меньше файлов.

In [ ]:
from spark_lab import timed

with timed("plain read with date filter"):
    print(spark.read.parquet(EVENTS).where("event_date = '2026-01-10'").count())

with timed("partitioned read with date filter"):
    print(spark.read.parquet(EVENTS_PARTITIONED).where("event_date = '2026-01-10'").count())

## Skew

На skewed key часть task'ов будет заметно дольше. Это хороший сценарий для объяснения, почему “добавить ресурсов” не всегда лечит проблему.

In [ ]:
benchmark_skew(spark, SKEWED)

In [ ]:
spark.stop()